In [1]:
import glob
import json

import pandas as pd

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
files = glob.glob("NTO workshop LAK26 Conrad PLUS/*.json")

In [4]:
data = [] 
for file in files:
    with open(file, "r", encoding="utf-8") as f:
        content = json.load(f)   # parse JSON → Python dict
        data.append(content)

In [5]:
df = pd.json_normalize(
    data,
    record_path="events",      
    meta=["metadata", "n_segments"],  
)

In [6]:
df.event_type.value_counts()

event_type
spoken_utterance    6063
mouse               2079
keyboard             634
inactivity           577
view_change          502
system               346
drawing              312
app_switch           215
page_scroll           84
Name: count, dtype: int64

In [7]:
df["text_combined"] = df["text"].combine_first(df["description"]).combine_first(df["details"])

In [ ]:
df[df['event_type']!='spoken_utterance']

# Clustering

In [11]:
import numpy as np
import pandas as pd
from sklearn.cluster import HDBSCAN
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

type_col = "event_type"

content_cols = [
    "text_combined",
    "text",
    "description",
    "details",
    "action",
    "target_desc",
    "tool",
    "from_app",
    "from_window",
    "to_app",
    "to_window",
    "direction",
    "delta_pixels",
]

def clean_val(v):
    if pd.isna(v):
        return ""
    v = str(v).strip()
    if v.lower() in ["nan", "none", "null"]:
        return ""
    return v

def row_content(r):
    parts = []

    for col in content_cols:
        if col in r.index:
            v = clean_val(r[col])
            if v:
                parts.append(v)

    if "x" in r.index and "y" in r.index:
        x = clean_val(r["x"])
        y = clean_val(r["y"])
        if x and y:
            parts.append(f"xy=({x},{y})")

    return " | ".join(dict.fromkeys(parts))

def make_windows(df, spoken_only=True, window=3):
    x = df.copy()

    if spoken_only:
        x = x[x[type_col] == "spoken_utterance"]

    x["content"] = x.apply(row_content, axis=1)
    x = x[x["content"].str.len() > 0].reset_index(drop=True)

    types = x[type_col].fillna("").astype(str).tolist()
    content = x["content"].tolist()

    rows = []
    for i in range(len(content) - window + 1):
        window_text = " ".join(
            f"[{types[j]}] {content[j]}" for j in range(i, i + window)
        )
        rows.append({"window_text": window_text})

    return (
        pd.DataFrame(rows)
        .drop_duplicates("window_text")
        .reset_index(drop=True)
    )

def cluster_windows(w, model_name="all-MiniLM-L6-v2", min_cluster_size=5):
    model = SentenceTransformer(model_name)

    X = model.encode(
        w["window_text"].tolist(),
        normalize_embeddings=True,
        show_progress_bar=True
    )

    h = HDBSCAN(
        min_cluster_size=min_cluster_size,
        metric="euclidean",
        copy=True
    )

    out = w.copy()
    out["cluster"] = h.fit_predict(X)

    return out, X

def cluster_counts(c):
    return (
        c[c.cluster != -1]
        .groupby("cluster")
        .size()
        .reset_index(name="n")
        .sort_values("n", ascending=False)
    )

def top_examples(c, X, top_n=10):
    rows = []

    for cl in sorted(c.cluster.unique()):
        if cl == -1:
            continue

        idx = c.index[c.cluster == cl].to_numpy()
        S = X[idx]

        center = S.mean(axis=0, keepdims=True)
        sims = cosine_similarity(S, center).ravel()

        for rank, pos in enumerate(np.argsort(-sims)[:top_n], 1):
            rows.append({
                "cluster": cl,
                "rank": rank,
                "similarity": sims[pos],
                "window_text": c.loc[idx[pos], "window_text"]
            })

    return pd.DataFrame(rows)

results = {}

with pd.ExcelWriter("sbert_clustering_results.xlsx", engine="openpyxl") as writer:
    for name, spoken_only in [("spoken_only", True), ("all_types", False)]:
        w = make_windows(df, spoken_only=spoken_only, window=3)
        c, X = cluster_windows(w, min_cluster_size=5)

        counts = cluster_counts(c)
        examples = top_examples(c, X, top_n=10)

        results[name] = {
            "windows": w,
            "clustered": c,
            "cluster_counts": counts,
            "top_examples": examples,
            "n_clusters": len(counts)
        }

        examples.to_excel(writer, sheet_name=name, index=False)

        print(name)
        print("windows:", len(w))
        print("clusters:", len(counts))

/Users/cborcher/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 11280.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|████████████████████████████████| 133/133 [00:03<00:00, 36.44it/s]


spoken_only
windows: 4254
clusters: 93


Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 11304.81it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|████████████████████████████████| 241/241 [00:04<00:00, 57.42it/s]


all_types
windows: 7697
clusters: 211
